# Mojo + MAX on Qualcomm IQ Devices

**Goal:** Run Mojo application logic on the IQ9/IQ8 ARM CPU and connect to a MAX or Genie inference backend for model execution.

**Architecture:**
```
Mojo app (ARM64 CPU)
  ↓  IPC / Unix socket
MAX graph or C++ Genie service (ARM64 CPU)
  ↓
QNN HTP backend
  ↓
Hexagon NPU
```

Mojo does **not** compile directly to the Hexagon NPU. It handles application logic, agent state, and tool routing on the ARM CPU.

## 1. Environment Check

In [ ]:
import subprocess, platform, shutil

print("Platform:", platform.platform())
print("Machine: ", platform.machine())

for tool in ["mojo", "magic", "uv"]:
    path = shutil.which(tool)
    if path:
        result = subprocess.run([tool, "--version"], capture_output=True, text=True)
        print(f"{tool}: {result.stdout.strip() or result.stderr.strip()}")
    else:
        print(f"{tool}: not found")

## 2. Hello Mojo

Write and run a minimal Mojo program on the current machine.

In [ ]:
hello_src = '''
fn main():
    print("Hello from Mojo on", String(sys.info.os_is_linux()) if True else "Windows/macOS")
'''

# Simpler version that always works:
hello_src = 'fn main():\n    print("Hello from Mojo")\n'

with open("hello.mojo", "w") as f:
    f.write(hello_src)

result = subprocess.run(["mojo", "run", "hello.mojo"], capture_output=True, text=True)
print(result.stdout or result.stderr)

## 3. MAX Graph: Simple Tensor Operation

Use MAX to build and execute a small computation graph (matrix multiply) on the available device.

In [ ]:
# Install MAX if not present
try:
    import max
    print("MAX already available")
except ImportError:
    subprocess.run(["uv", "add", "max"], check=True)
    import max
    print("MAX installed")

In [ ]:
import numpy as np
from max.graph import Graph, TensorType
from max import engine
from max.dtype import DType

# Build a simple matmul graph
with Graph("matmul", input_types=[
    TensorType(DType.float32, shape=[4, 4]),
    TensorType(DType.float32, shape=[4, 4]),
]) as graph:
    a, b = graph.inputs
    out = a @ b
    graph.output(out)

# Run it
session = engine.InferenceSession()
model = session.load(graph)

A = np.eye(4, dtype=np.float32) * 2
B = np.ones((4, 4), dtype=np.float32)

result = model.execute(input0=A, input1=B)
print("Result (2*I @ ones(4,4)):")
print(result[0])

## 4. On-Device: Connect to Genie NPU Service (IQ9/IQ8)

When running on the IQ9, Mojo (or Python) talks to the Genie server over a local socket.
The Genie server (`../../EdgeAgent/genie_server.py`) exposes an OpenAI-compatible `/v1/chat/completions` endpoint.

This cell assumes the Genie server is already running on port 8081.

In [ ]:
import urllib.request, json

GENIE_URL = "http://127.0.0.1:8081/v1/chat/completions"  # change host if calling from PC
MODEL_NAME = "gemma-4-e4b-genie"

payload = json.dumps({
    "model": MODEL_NAME,
    "messages": [{"role": "user", "content": "Explain edge AI in two sentences."}],
    "max_tokens": 64,
    "stream": False,
}).encode()

try:
    req = urllib.request.Request(
        GENIE_URL,
        data=payload,
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=30) as resp:
        data = json.loads(resp.read())
    print(data["choices"][0]["message"]["content"])
except Exception as e:
    print(f"Genie service not reachable: {e}")
    print("Start it first: see EdgeAgent/start_iq9_genie.sh")

## 5. Next Steps

- **Agent loop in Mojo** — prompt builder → Genie call → tool parse → tool execute → feed back
- **MAX model loading** — load a quantized model via `engine.InferenceSession` directly (CPU path)
- **NPU via Genie** — C++ service wrapping Genie API; Mojo calls it over Unix socket
- **ROS 2 integration** — sensor inputs → Mojo state → Gemma reasoning → actuator commands

See `../EdgeAgent/` for the Genie server, agent loop, and IQ9 bring-up scripts.